# Chocolate Sales Analytics: Data Cleaning and Analysis

**Portfolio project for a  Data Analyst**



## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Load the raw dataset




In [2]:
raw_path = Path("../data/Chocolate Sales.csv")

if not raw_path.is_file():
    raise FileNotFoundError(
        "Raw dataset not found. Place 'Chocolate Sales.csv' in the project's "
        "'data/' folder, then run this notebook from the 'python/' folder."
    )

raw_df = pd.read_csv(raw_path)
df = raw_df.copy(deep=True)

print("Raw dataset loaded successfully.")
print("Source file:", raw_path.as_posix())
print("Shape:", raw_df.shape)


Raw dataset loaded successfully.\nSource file: ../data/Chocolate Sales.csv\nShape: (1094, 6)\n

## 3. Initial inspection

In [3]:
display(raw_df.head())
display(raw_df.tail())
print("Shape (rows, columns):", raw_df.shape)
print("\nColumn names:")
print(raw_df.columns.tolist())
print("\nData types:")
print(raw_df.dtypes)
print("\nDescriptive statistics (all columns):")
display(raw_df.describe(include="all").T)

,Sales Person,Country,Product,Date,Amount,Boxes Shipped
0,Jehu Rudeforth,UK,Mint Chip Choco,04-Jan-22,"$5,320",180
1,Van Tuxwell,India,85% Dark Bars,01-Aug-22,"$7,896",94
2,Gigi Bohling,India,Peanut Butter Cubes,07-Jul-22,"$4,501",91
3,Jan Morforth,Australia,Peanut Butter Cubes,27-Apr-22,"$12,726",342
4,Jehu Rudeforth,UK,Peanut Butter Cubes,24-Feb-22,"$13,685",184


,Sales Person,Country,Product,Date,Amount,Boxes Shipped
1089,Karlen McCaffrey,Australia,Spicy Special Slims,17-May-22,"$4,410",323
1090,Jehu Rudeforth,USA,White Choc,07-Jun-22,"$6,559",119
1091,Ches Bonnell,Canada,Organic Choco Syrup,26-Jul-22,$574,217
1092,Dotty Strutley,India,Eclairs,28-Jul-22,"$2,086",384
1093,Karlen McCaffrey,India,70% Dark Bites,23-May-22,"$5,075",344


Shape (rows, columns): (1094, 6)

Column names:
['Sales Person', 'Country', 'Product', 'Date', 'Amount', 'Boxes Shipped']

Data types:
Sales Person     object
Country          object
Product          object
Date             object
Amount           object
Boxes Shipped     int64
dtype: object

Descriptive statistics (all columns):


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Sales Person,1094,25,Kelci Walkden,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,1094,6,Australia,205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,1094,22,Eclairs,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,1094,168,30-Jun-22,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Amount,1094,827,"$2,317",5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Boxes Shipped,1094.0,NaN,NaN,NaN,161.797989,121.544145,1.0,70.0,135.0,228.75,709.0


### 3.1 Unique values and data-quality checks

In [4]:
print("Unique values per column:")
display(raw_df.nunique(dropna=False).rename("unique_values").to_frame())

print("Missing values per column:")
display(raw_df.isna().sum().rename("missing_values").to_frame())

# Blank strings and leading/trailing whitespace in text-like columns
text_cols = raw_df.select_dtypes(include=["object", "string"]).columns
quality_rows = []
for col in text_cols:
    s = raw_df[col].astype("string")
    quality_rows.append({
        "column": col,
        "blank_strings": int((s.str.strip() == "").fillna(False).sum()),
        "leading_or_trailing_whitespace": int((s.notna() & s.ne(s.str.strip())).sum()),
        "unique_values": int(raw_df[col].nunique(dropna=False))
    })
display(pd.DataFrame(quality_rows))

print("Exact duplicate rows:", raw_df.duplicated().sum())

Unique values per column:


,unique_values
Sales Person,25
Country,6
Product,22
Date,168
Amount,827
Boxes Shipped,385


Missing values per column:


,missing_values
Sales Person,0
Country,0
Product,0
Date,0
Amount,0
Boxes Shipped,0


,column,blank_strings,leading_or_trailing_whitespace,unique_values
0,Sales Person,0,0,25
1,Country,0,0,6
2,Product,0,0,22
3,Date,0,0,168
4,Amount,0,1094,827


Exact duplicate rows: 0


### 3.2 Inspect dates, numeric fields, and categories

In [5]:
# Parse using the date format observed in this dataset: day-abbreviated-month-two-digit-year.
raw_dates = pd.to_datetime(raw_df["Date"].astype("string").str.strip(),
                           format="%d-%b-%y", errors="coerce")
print("Invalid/unparseable dates:", int(raw_dates.isna().sum()))
print("Date range:", raw_dates.min(), "to", raw_dates.max())

# Inspect currency values before converting them.
amount_text = raw_df["Amount"].astype("string")
amount_numeric = pd.to_numeric(
    amount_text.str.replace(r"[$,\s]", "", regex=True),
    errors="coerce"
)
print("Amount values that fail numeric conversion:", int(amount_numeric.isna().sum()))
print("Amount minimum/maximum:", amount_numeric.min(), "/", amount_numeric.max())
print("Amount zero/negative counts:", int((amount_numeric == 0).sum()), "/", int((amount_numeric < 0).sum()))

boxes_numeric = pd.to_numeric(raw_df["Boxes Shipped"], errors="coerce")
print("Boxes Shipped conversion failures:", int(boxes_numeric.isna().sum()))
print("Boxes Shipped zero/negative counts:",
      int((boxes_numeric == 0).sum()), "/", int((boxes_numeric < 0).sum()))

for col in ["Country", "Product", "Sales Person"]:
    print(f"\\n{col} values ({raw_df[col].nunique()} unique):")
    print(sorted(raw_df[col].dropna().astype(str).unique().tolist()))

Invalid/unparseable dates: 0
Date range: 2022-01-03 00:00:00 to 2022-08-31 00:00:00
Amount values that fail numeric conversion: 0
Amount minimum/maximum: 7 / 22050
Amount zero/negative counts: 0 / 0
Boxes Shipped conversion failures: 0
Boxes Shipped zero/negative counts: 0 / 0
\nCountry values (6 unique):
['Australia', 'Canada', 'India', 'New Zealand', 'UK', 'USA']
\nProduct values (22 unique):
['50% Dark Bites', '70% Dark Bites', '85% Dark Bars', '99% Dark & Pure', 'After Nines', 'Almond Choco', "Baker's Choco Chips", 'Caramel Stuffed Bars', 'Choco Coated Almonds', 'Drinking Coco', 'Eclairs', 'Fruit & Nut Bars', 'Manuka Honey Choco', 'Milk Bars', 'Mint Chip Choco', 'Orange Choco', 'Organic Choco Syrup', 'Peanut Butter Cubes', 'Raspberry Choco', 'Smooth Sliky Salty', 'Spicy Special Slims', 'White Choc']
\nSales Person values (25 unique):
['Andria Kimpton', 'Barr Faughny', 'Beverie Moffet', 'Brien Boise', 'Camilla Castle', 'Ches Bonnell', 'Curtice Advani', 'Dennison Crosswaite', 'Dotty 

## 4. Clean the data

In [6]:
# Work on a copy; raw_df remains unchanged.
clean_df = raw_df.copy(deep=True)

# Standardize column names to lowercase snake_case.
clean_df.columns = (
    clean_df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Trim whitespace from text columns only.
for col in ["sales_person", "country", "product"]:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# Convert dates using the observed format; invalid dates become NaT for visibility.
clean_df["date"] = pd.to_datetime(
    clean_df["date"].astype("string").str.strip(),
    format="%d-%b-%y",
    errors="coerce"
)

# Remove currency symbols, thousands separators, and whitespace from Amount.
clean_df["amount"] = pd.to_numeric(
    clean_df["amount"].astype("string").str.replace(r"[$,\s]", "", regex=True),
    errors="coerce"
)

# Validate Boxes Shipped as a numeric whole-number field before using nullable integer type.
boxes = pd.to_numeric(clean_df["boxes_shipped"], errors="coerce")
non_integer_boxes = boxes.notna() & (boxes % 1 != 0)
if non_integer_boxes.any():
    print("Warning: non-integer Boxes Shipped values found; inspect these rows:")
    display(clean_df.loc[non_integer_boxes])
clean_df["boxes_shipped"] = boxes.astype("Int64")

print("Cleaning steps completed.")
display(clean_df.head())
print(clean_df.dtypes)

Cleaning steps completed.


,sales_person,country,product,date,amount,boxes_shipped
0,Jehu Rudeforth,UK,Mint Chip Choco,2022-01-04,5320,180
1,Van Tuxwell,India,85% Dark Bars,2022-08-01,7896,94
2,Gigi Bohling,India,Peanut Butter Cubes,2022-07-07,4501,91
3,Jan Morforth,Australia,Peanut Butter Cubes,2022-04-27,12726,342
4,Jehu Rudeforth,UK,Peanut Butter Cubes,2022-02-24,13685,184


sales_person     string[python]
country          string[python]
product          string[python]
date             datetime64[ns]
amount                    Int64
boxes_shipped             Int64
dtype: object


## 5. Validate the cleaned dataset

In [7]:
print("Cleaned shape:", clean_df.shape)
print("Raw row count:", len(raw_df))
print("Cleaned row count:", len(clean_df))
print("Row count preserved:", len(raw_df) == len(clean_df))

print("\nMissing values:")
display(clean_df.isna().sum().rename("missing_values").to_frame())

text_cols_clean = clean_df.select_dtypes(include=["object", "string"]).columns
blank_check = {
    col: int((clean_df[col].astype("string").str.strip() == "").fillna(False).sum())
    for col in text_cols_clean
}
print("\nBlank strings in remaining text columns:")
print(blank_check)

print("\nExact duplicate rows:", clean_df.duplicated().sum())
print("Invalid dates:", int(clean_df["date"].isna().sum()))
print("Invalid amounts:", int(clean_df["amount"].isna().sum()))
print("Invalid Boxes Shipped:", int(clean_df["boxes_shipped"].isna().sum()))
print("Zero/negative amounts:", int(((clean_df["amount"] <= 0) & clean_df["amount"].notna()).sum()))
print("Zero/negative boxes:", int(((clean_df["boxes_shipped"] <= 0) & clean_df["boxes_shipped"].notna()).sum()))

print("\nCleaned descriptive statistics:")
display(clean_df.describe(include="all").T)

Cleaned shape: (1094, 6)
Raw row count: 1094
Cleaned row count: 1094
Row count preserved: True

Missing values:


,missing_values
sales_person,0
country,0
product,0
date,0
amount,0
boxes_shipped,0



Blank strings in remaining text columns:
{'sales_person': 0, 'country': 0, 'product': 0}

Exact duplicate rows: 0
Invalid dates: 0
Invalid amounts: 0
Invalid Boxes Shipped: 0
Zero/negative amounts: 0
Zero/negative boxes: 0

Cleaned descriptive statistics:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
sales_person,1094,25,Kelci Walkden,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,1094,6,Australia,205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product,1094,22,Eclairs,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,1094,NaN,NaN,NaN,2022-05-03 09:04:56.160877568,2022-01-03 00:00:00,2022-03-02 00:00:00,2022-05-11 00:00:00,2022-07-04 00:00:00,2022-08-31 00:00:00,NaN
amount,1094.0,<NA>,<NA>,<NA>,5652.308044,7.0,2390.5,4868.5,8027.25,22050.0,4102.442014
boxes_shipped,1094.0,<NA>,<NA>,<NA>,161.797989,1.0,70.0,135.0,228.75,709.0,121.544145


### 5.1 Check that valid values were preserved

In [8]:
# Compare original rows with the cleaned data after applying only the intended formatting/type conversions.
# The columns have been renamed, so compare values after equivalent conversion.
expected = raw_df.copy(deep=True)
expected.columns = (
    expected.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
)
for col in ["sales_person", "country", "product"]:
    expected[col] = expected[col].astype("string").str.strip()
expected["date"] = pd.to_datetime(
    expected["date"].astype("string").str.strip(),
    format="%d-%b-%y", errors="coerce"
)
expected["amount"] = pd.to_numeric(
    expected["amount"].astype("string").str.replace(r"[$,\s]", "", regex=True),
    errors="coerce"
)
expected["boxes_shipped"] = pd.to_numeric(expected["boxes_shipped"], errors="coerce").astype("Int64")

# This should be True when the cleaning logic is deterministic and no rows were removed/reordered.
print("Cleaned data matches expected conversions:", clean_df.equals(expected))
print("Rows removed:", len(raw_df) - len(clean_df))

Cleaned data matches expected conversions: True
Rows removed: 0


## 6. Pandas sales analysis

### 6.1 Overall totals and sales by country, product, and salesperson

In [9]:
# Overall totals
total_sales_amount = clean_df["amount"].sum()
total_boxes_shipped = clean_df["boxes_shipped"].sum()

print(f"Total sales amount: ${total_sales_amount:,.2f}")
print(f"Total boxes shipped: {total_boxes_shipped:,}")

# Sales by dimension
sales_by_country = clean_df.groupby("country", as_index=False)["amount"].sum().sort_values("amount", ascending=False)
sales_by_product = clean_df.groupby("product", as_index=False)["amount"].sum().sort_values("amount", ascending=False)
sales_by_salesperson = clean_df.groupby("sales_person", as_index=False)["amount"].sum().sort_values("amount", ascending=False)

print("\nSales by country:")
display(sales_by_country)
print("\nSales by product (top 10):")
display(sales_by_product.head(10))
print("\nSales by salesperson (top 10):")
display(sales_by_salesperson.head(10))

Total sales amount: $6,183,625.00
Total boxes shipped: 177,007

Sales by country:


,country,amount
0,Australia,1137367
4,UK,1051792
2,India,1045800
5,USA,1035349
1,Canada,962899
3,New Zealand,950418



Sales by product (top 10):


,product,amount
19,Smooth Sliky Salty,349692
0,50% Dark Bites,341712
21,White Choc,329147
17,Peanut Butter Cubes,324842
10,Eclairs,312445
3,99% Dark & Pure,299796
2,85% Dark Bars,299229
16,Organic Choco Syrup,294700
20,Spicy Special Slims,293454
14,Mint Chip Choco,283969



Sales by salesperson (top 10):


,sales_person,amount
5,Ches Bonnell,320901
20,Oby Sorrel,316645
17,Madelene Upcott,316099
3,Brien Boise,312816
16,Kelci Walkden,311710
23,Van Tuxwell,303149
7,Dennison Crosswaite,291669
2,Beverie Moffet,278922
14,Kaine Padly,266490
19,Marney O'Breen,259742


### 6.2 Monthly sales trend and top performers (Jan–Aug 2022)

In [10]:
# Monthly trend. Month is based on the parsed transaction date.
monthly_sales = (
    clean_df.assign(month=clean_df["date"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)["amount"].sum()
    .sort_values("month")
)
print("Monthly sales amount:")
display(monthly_sales)

print("\nTop-performing country by sales amount:")
display(sales_by_country.head(1))
print("\nTop-performing product by sales amount:")
display(sales_by_product.head(1))
print("\nTop-performing salesperson by sales amount:")
display(sales_by_salesperson.head(1))

Monthly sales amount:


,month,amount
0,2022-01,896105
1,2022-02,699377
2,2022-03,749483
3,2022-04,674051
4,2022-05,752892
5,2022-06,865144
6,2022-07,803425
7,2022-08,743148



Top-performing country by sales amount:


,country,amount
0,Australia,1137367



Top-performing product by sales amount:


,product,amount
19,Smooth Sliky Salty,349692



Top-performing salesperson by sales amount:


,sales_person,amount
5,Ches Bonnell,320901


## 7. Export the cleaned CSV



In [11]:
# Save the cleaned dataset in the project's data/ folder.
project_data_dir = Path("../data")

if not project_data_dir.is_dir():
    raise FileNotFoundError(
        "Project data folder not found. Run this notebook from the project's "
        "'python/' folder and make sure the 'data/' folder exists."
    )

output_path = project_data_dir / "chocolate_sales_cleaned.csv"
clean_df.to_csv(output_path, index=False, date_format="%Y-%m-%d")

print("Cleaned CSV exported successfully.")
print("Output file:", output_path.as_posix())
print("Rows exported:", len(clean_df))
print("Columns exported:", len(clean_df.columns))


Cleaned CSV exported successfully.\nOutput file: ../data/chocolate_sales_cleaned.csv\nRows exported: 1094\nColumns exported: 6\n